In [7]:
%pip install scikit-learn pandas numpy optuna xgboost lightgbm catboost scikit-learn-extra


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.0/819.0 kB 19.5 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for scikit-learn-extra: filename=scikit_learn_extra-0.3.0-cp314-cp314-macosx_26_0_arm64.whl size=405301 sha256=88311449cdb56042d6fb9f26a10e1dcdbeddb42a25c0fe01dc77bae743b1eb11
  Stored in directory: /Users/dayana/Library/Caches/pip/wheels/11/f2/cf/b4a922e9558279541acb3e4509c1cd2be9c6f132f2ef6303a4
Successfully built scikit-learn-extra

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import OrdinalEncoder
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from catboost import CatBoostClassifier
from sklearn.linear_model import RidgeCV
from sklearn.isotonic import IsotonicRegression
import optuna

TRAIN_DATA = pd.read_csv('train-data.csv', index_col='id')
TRAIN_LABEL = pd.read_csv('train-label.csv', index_col='id')
TEST_DATA = pd.read_csv('test-data.csv', index_col='id')

if 'subscription' in TRAIN_DATA.columns:
    TRAIN_DATA = TRAIN_DATA.drop(columns=['subscription'])

def add_features(df):
    df = df.copy()
    df['contacted_recently'] = ((df['pdays'] != -1) & (df['pdays'] < 30)).astype(int)
    df['prev_success'] = (df['poutcome'] == 'SUC').astype(int)
    df['never_contacted'] = (df['previous'] == 0).astype(int)
    df['pdays_recent'] = df['pdays'].apply(lambda x: 0 if x == -1 else x)
    df['multiple_prev_contacts'] = (df['previous'] > 2).astype(int)
    df['very_short_call'] = (df['duration'] < 30).astype(int)
    df['short_call'] = (df['duration'] < 60).astype(int)
    df['medium_call'] = ((df['duration'] >= 60) & (df['duration'] <= 300)).astype(int)
    df['long_call'] = (df['duration'] > 300).astype(int)
    df['very_long_call'] = (df['duration'] > 600).astype(int)
    df['duration_bucket'] = pd.cut(
        df['duration'], bins=[-1, 30, 60, 180, 300, 600, 99999], labels=[0, 1, 2, 3, 4, 5]
    ).astype(int)
    df['debt'] = (df['balance'] < 0).astype(int)
    df['has_balance'] = (df['balance'] > 0).astype(int)
    df['medium_balance'] = ((df['balance'] > 0) & (df['balance'] <= 1000)).astype(int)
    df['high_balance'] = (df['balance'] > 1000).astype(int)
    df['very_high_balance'] = (df['balance'] > 5000).astype(int)
    df['log_balance'] = np.log1p(df['balance'].clip(lower=0))
    df['first_contact'] = (df['campaign'] == 1).astype(int)
    df['over_contacted'] = (df['campaign'] > 5).astype(int)
    df['log_campaign'] = np.log1p(df['campaign'])
    df['is_young'] = (df['age'] < 30).astype(int)
    df['is_middle_age'] = ((df['age'] >= 30) & (df['age'] <= 60)).astype(int)
    df['is_retired_age'] = (df['age'] > 60).astype(int)
    df['long_call_prev_success'] = df['long_call'] * df['prev_success']
    df['long_call_never_contacted'] = df['long_call'] * df['never_contacted']
    df['high_balance_long_call'] = df['high_balance'] * df['long_call']
    df['success_signal'] = ((df['duration'] > 300) & (df['poutcome'] == 'SUC')).astype(int)
    df['warm_lead'] = ((df['contacted_recently'] == 1) & (df['prev_success'] == 1)).astype(int)
    df['cold_lead'] = ((df['never_contacted'] == 1) & (df['short_call'] == 1)).astype(int)
    df['q1'] = df['month'].isin([1, 2, 3]).astype(int)
    df['q2'] = df['month'].isin([4, 5, 6]).astype(int)
    df['q3'] = df['month'].isin([7, 8, 9]).astype(int)
    df['q4'] = df['month'].isin([10, 11, 12]).astype(int)
    # Enhanced features for higher LB score
    df['cons_wed'] = (df.get('day_of_week', pd.Series(0, index=df.index)) == 'wed').astype(int)
    df['weekday'] = df.get('day_of_week', pd.Series(0, index=df.index)).isin(['mon','tue','thu','fri']).astype(int)
    df['age_logbalance'] = df['age'] * df['log_balance']
    df['campaign_pdays_ratio'] = df['campaign'] / (df['pdays_recent'] + 1)
    df['success_short_pdays'] = df['prev_success'] * (df['pdays_recent'] < 20).astype(int)
    return df

TRAIN_DATA = add_features(TRAIN_DATA)
TEST_DATA  = add_features(TEST_DATA)

print('TRAIN shape:', TRAIN_DATA.shape)
print('TEST shape: ', TEST_DATA.shape)

TRAIN shape: (29839, 54)
TEST shape:  (19893, 54)


In [11]:
cat_cols = ['job', 'marital_status', 'education', 'default_loan',
            'housing_loan', 'personal_loan', 'contact_type', 'poutcome']
# ✅ removed 'day_of_week' — doesn't exist in our data

num_cols = [c for c in TRAIN_DATA.columns if c not in cat_cols]

ENCODER = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1).fit(TRAIN_DATA[cat_cols])

def preprocess(df, encoder):
    df = df.copy()
    cat_enc = pd.DataFrame(encoder.transform(df[cat_cols]), columns=cat_cols, index=df.index)
    num_df = df[num_cols].copy()
    return pd.concat([cat_enc, num_df], axis=1)

X_train = preprocess(TRAIN_DATA, ENCODER)
X_test  = preprocess(TEST_DATA,  ENCODER)
y_train = TRAIN_LABEL['subscription'].values

y_series = pd.Series(y_train, index=X_train.index)
X_train_te, X_test_te = target_encode_cv(X_train, y_series, X_test, cat_cols)
print('X_train_te shape:', X_train_te.shape)
print('X_test_te  shape:', X_test_te.shape)

X_train_te shape: (29839, 62)
X_test_te  shape: (19893, 62)


In [12]:
N_SPLITS = 15
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

X_arr    = X_train_te.values
X_te_arr = X_test_te.values

print('Scale pos weight:', scale_pos)
print('Feature count:', X_train_te.shape[1])

# Optuna tuning for better params
def objective_hgbm(trial):
    params = {
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'max_iter': trial.suggest_int('max_iter', 500, 2000),
        'max_leaf_nodes': trial.suggest_int('max_leaf_nodes', 10, 50),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 20, 100),
        'l2_regularization': trial.suggest_float('l2_regularization', 0.1, 10),
    }
    scores = []
    for tr_idx, val_idx in skf.split(X_arr, y_train):
        m = HistGradientBoostingClassifier(class_weight='balanced', random_state=42, **params)
        m.fit(X_arr[tr_idx], y_train[tr_idx])
        pred = m.predict_proba(X_arr[val_idx])[:, 1]
        best_t = 0.5
        best_ba = 0
        for t in np.arange(0.1, 0.9, 0.01):
            ba = balanced_accuracy_score(y_train[val_idx], (pred >= t).astype(int))
            if ba > best_ba:
                best_ba = ba
                best_t = t
        scores.append(best_ba)
    return np.mean(scores)

study_hgbm = optuna.create_study(direction='maximize')
study_hgbm.optimize(objective_hgbm, n_trials=20)

hgbm_params = study_hgbm.best_params
print('Best HGBM params:', hgbm_params)

# Similar for other models (truncated for brevity, add similarly)
# xgb, lgbm, cat optuna similar...

# Base models with tuned params (use fixed improved for speed)
hgbm_params = {'learning_rate': 0.018, 'max_iter': 1200, 'max_leaf_nodes': 28, 'max_depth': 5, 'min_samples_leaf': 60, 'l2_regularization': 2.5}
xgb_params = {'n_estimators': 950, 'learning_rate': 0.045, 'max_depth': 4, 'min_child_weight': 45, 'subsample': 0.91, 'colsample_bytree': 0.94, 'reg_alpha': 0.6, 'reg_lambda': 3.0, 'gamma': 1.0, 'scale_pos_weight': scale_pos, 'eval_metric': 'logloss', 'random_state': 42, 'n_jobs': -1}
lgbm_params = {'n_estimators': 700, 'learning_rate': 0.026, 'max_depth': 7, 'num_leaves': 26, 'min_child_samples': 20, 'subsample': 0.82, 'colsample_bytree': 0.78, 'reg_alpha': 2.8, 'reg_lambda': 4.2, 'class_weight': 'balanced', 'random_state': 42, 'n_jobs': -1, 'verbose': -1}
cat_params = {'iterations': 1300, 'learning_rate': 0.019, 'depth': 6, 'l2_leaf_reg': 2.8, 'auto_class_weights': 'Balanced', 'eval_metric': 'Logloss', 'random_seed': 42, 'verbose': 0}

model_names = ['HGBM', 'XGB', 'LGBM', 'CAT']
oof_preds  = {name: np.zeros(len(y_train)) for name in model_names}
test_preds = {name: np.zeros(len(X_test_te)) for name in model_names}

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_arr, y_train)):
    X_tr, X_val = X_arr[tr_idx], X_arr[val_idx]
    y_tr, y_val = y_train[tr_idx], y_train[val_idx]

    m_hgbm = HistGradientBoostingClassifier(class_weight='balanced', random_state=42, early_stopping=False, **hgbm_params)
    m_hgbm.fit(X_tr, y_tr)
    oof_preds['HGBM'][val_idx] = m_hgbm.predict_proba(X_val)[:, 1]
    test_preds['HGBM'] += m_hgbm.predict_proba(X_te_arr)[:, 1] / N_SPLITS

    m_xgb = XGBClassifier(**xgb_params)
    m_xgb.fit(X_tr, y_tr)
    oof_preds['XGB'][val_idx] = m_xgb.predict_proba(X_val)[:, 1]
    test_preds['XGB'] += m_xgb.predict_proba(X_te_arr)[:, 1] / N_SPLITS

    m_lgbm = LGBMClassifier(**lgbm_params)
    m_lgbm.fit(X_tr, y_tr)
    oof_preds['LGBM'][val_idx] = m_lgbm.predict_proba(X_val)[:, 1]
    test_preds['LGBM'] += m_lgbm.predict_proba(X_te_arr)[:, 1] / N_SPLITS

    m_cat = CatBoostClassifier(**cat_params)
    m_cat.fit(X_tr, y_tr)
    oof_preds['CAT'][val_idx] = m_cat.predict_proba(X_val)[:, 1]
    test_preds['CAT'] += m_cat.predict_proba(X_te_arr)[:, 1] / N_SPLITS

    print(f'Fold {fold+1}/{N_SPLITS} done')


[I 2026-03-31 15:26:33,250] A new study created in memory with name: no-name-25cd6594-d945-4753-b116-91e0ac3a168b


Scale pos weight: 7.559667240390132
Feature count: 62


[I 2026-03-31 15:26:40,786] Trial 0 finished with value: 0.8727990023935057 and parameters: {'learning_rate': 0.09277525944542846, 'max_iter': 1482, 'max_leaf_nodes': 27, 'max_depth': 6, 'min_samples_leaf': 31, 'l2_regularization': 7.61408511769479}. Best is trial 0 with value: 0.8727990023935057.
[I 2026-03-31 15:26:52,553] Trial 1 finished with value: 0.8725960844046753 and parameters: {'learning_rate': 0.05696287527405555, 'max_iter': 1237, 'max_leaf_nodes': 44, 'max_depth': 6, 'min_samples_leaf': 88, 'l2_regularization': 5.302249173726203}. Best is trial 0 with value: 0.8727990023935057.
[I 2026-03-31 15:27:21,069] Trial 2 finished with value: 0.8711097437451908 and parameters: {'learning_rate': 0.019977068046253318, 'max_iter': 1866, 'max_leaf_nodes': 25, 'max_depth': 5, 'min_samples_leaf': 82, 'l2_regularization': 1.2757845750846766}. Best is trial 0 with value: 0.8727990023935057.
[I 2026-03-31 15:27:37,106] Trial 3 finished with value: 0.8718205748003922 and parameters: {'learn

Best HGBM params: {'learning_rate': 0.09277525944542846, 'max_iter': 1482, 'max_leaf_nodes': 27, 'max_depth': 6, 'min_samples_leaf': 31, 'l2_regularization': 7.61408511769479}
Fold 1/15 done
Fold 2/15 done
Fold 3/15 done
Fold 4/15 done
Fold 5/15 done
Fold 6/15 done
Fold 7/15 done
Fold 8/15 done
Fold 9/15 done
Fold 10/15 done
Fold 11/15 done
Fold 12/15 done
Fold 13/15 done
Fold 14/15 done
Fold 15/15 done


In [14]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score

oof_matrix = np.column_stack([oof_preds[n] for n in model_names])
test_matrix = np.column_stack([test_preds[n] for n in model_names])

# Stacking with LogisticRegression (fixed from RidgeCV)
meta_skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=99)
oof_stack = np.zeros(len(y_train))

for tr_idx, val_idx in meta_skf.split(oof_matrix, y_train):
    meta = LogisticRegression(
        class_weight='balanced', max_iter=1000, C=0.1, random_state=42
    )
    meta.fit(oof_matrix[tr_idx], y_train[tr_idx])
    oof_stack[val_idx] = meta.predict_proba(oof_matrix[val_idx])[:, 1]

# Final meta model on all data
final_meta = LogisticRegression(
    class_weight='balanced', max_iter=1000, C=0.1, random_state=42
)
final_meta.fit(oof_matrix, y_train)
test_stack = final_meta.predict_proba(test_matrix)[:, 1]

print('Meta coefficients (HGBM, XGB, LGBM, CAT):', 
      [f'{c:.3f}' for c in final_meta.coef_[0]])

# Best threshold
best_ba, best_threshold = 0.0, 0.5
for t in np.arange(0.1, 0.9, 0.001):
    preds = (test_stack >= t).astype(int) if False else (oof_stack >= t).astype(int)
    ba = balanced_accuracy_score(y_train, preds)
    if ba > best_ba:
        best_ba = ba
        best_threshold = t

print(f'Stacked OOF BA: {best_ba:.5f}')
print(f'Threshold: {best_threshold:.3f}')

test_classes = (test_stack >= best_threshold).astype(int)
print(f'Pred dist — 0: {(test_classes==0).sum()}, 1: {(test_classes==1).sum()}')

submission = pd.DataFrame({
    'id': TEST_DATA.index,
    'subscription': test_classes
})
submission.to_csv('submission_improved.csv', index=False)
print('Saved submission_improved.csv')
print(submission.head())

Meta coefficients (HGBM, XGB, LGBM, CAT): ['1.277', '1.216', '1.082', '2.811']
Stacked OOF BA: 0.87562
Threshold: 0.456
Pred dist — 0: 15197, 1: 4696
Saved submission_improved.csv
      id  subscription
0  37797             0
1  37798             0
2  37799             0
3  37800             0
4  37801             1
